In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
files_path = 'SOH raw live run files'
os.listdir(files_path)

FileNotFoundError: [Errno 2] No such file or directory: 'SOH raw live run files'

In [4]:
files_ = os.listdir(files_path)

In [5]:
files_ = files_[1]
files_

"SOH - 31-Jan'26 1.xlsb"

In [3]:
# df = pd.DataFrame()

# for file in files_:
#     tmp = pd.read_excel(f'{files_path}/{file}', sheet_name=None)
#     print(f'{file}: {tmp.keys()}')

#     if 'Sheet1' in tmp.keys():
#         file_df = tmp['Sheet1']
#         file_df.columns = file_df.columns.str.lower()
#     else:
#         file_df = tmp['Base']
#         file_df.columns = file_df.columns.str.lower()

#     file_df['file'] = file
#     df = pd.concat([df, file_df], ignore_index=True)

df = pd.read_excel(f"/data/aman_singh/acuuracy_check/SOH - 01 Jun.xlsx", sheet_name='Base')

In [4]:
df.columns

Index(['Date', 'Chain', 'FSN', 'SOH', 'material_code', 'Desc', 'Brand', 'UOM',
       'vol per unit', 'Vol', 'FY INDEX', 'Vol in KL', 'BPM in lacs',
       'Category', 'eCOM Brands', 'PSKU', 'PDES', 'Club SKU'],
      dtype='object')

In [5]:
df[df['Date'] != df['date']]

KeyError: 'date'

In [6]:
df.drop('date', axis=1, inplace=True)

KeyError: "['date'] not found in axis"

In [7]:
df.columns = df.columns.str.lower()

In [8]:
df

,date,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,vol in kl,bpm in lacs,category,ecom brands,psku,pdes,club sku
0,2026-06-01,Blinkit,10000059,4605,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,23.025000,169846.960103,23.025000,39.107263,Edible,eCOM Brands,718322,SAFF TOTAL 5L JAR,5LTR
1,2026-06-01,Blinkit,10000063,3867,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.146946,293725.254052,0.146946,0.431618,Core Foods,eCOM Brands,718494,SMO MSL&COR 38g PCH,30-45GM
2,2026-06-01,Blinkit,10000361,651,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,0.651000,25994.993781,0.651000,0.169227,Non Core Foods,eCOM Brands,807029,SAFF SALT 1kg PCH,1KG
3,2026-06-01,Blinkit,10000362,6438,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,900.0,5.794200,123682.266041,5.794200,7.166398,Edible,eCOM Brands,718328,SAFF TASTY 1L PCH,1LTR PC
4,2026-06-01,Blinkit,10000364,83758,720466,SAF ACTIV 1L PCH-LMRP MC,SAFF ACTV,KL,932.0,78.062456,118908.484774,78.062456,92.822884,Edible,eCOM Brands,718398,SAFF ACTIVE 1L PCH,1LTR PC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2795,2026-06-02,Amazon ARIPL,B0FS6GCRDG,2,721916,SAFF GOLD RC 4L JAR,SAFF GOLD,KL,4000.0,0.008000,137662.938527,0.008000,0.011013,Edible,eCOM Brands,722166,SAFF GOLD 4L JAR RC,4 LTR
2796,2026-06-02,Amazon ARIPL,B0FS6P6VTW,4,721914,SAF GOLD 4L JAR,SAFF GOLD,KL,4000.0,0.016000,137662.938527,0.016000,0.022026,Edible,eCOM Brands,722167,SAFF GOLD 4L JAR,4 LTR
2797,2026-06-02,Amazon ARIPL,B0G1SKX4CM,5408,811179,SAFF CDPRS CNO 1L,SAF_CDPRS,KL,1000.0,5.408000,330000.000000,5.408000,17.846400,Edible,eCOM Brands,811181,SAFFOLA COLDPRESS CNO 1L,1 LTR
2798,2026-06-02,Amazon ARIPL,B0GGHLBJL3,854,732780,SF HIGH PROTEIN OAT 1KG PC,SFOATS-PR,TO,1000.0,0.854000,430000.000000,0.854000,3.672200,Foods NPD,eCOM Brands,732778,SF HIGH PROTEIN OATS 1KG POUCH,1KG


In [9]:
soh_base = pd.read_csv('/data/aman_singh/acuuracy_check/soh_base_may_run.csv')
soh_base

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,vol_in_lit,vol in rum,indexrate,indexbpm,club sku,roum,roum divide,month,day,month.1
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84446,Amazon ARIPL,B0FGQB8RKZ,470.0,730698,COCO SOUL CLDPRS CNO 1L,CO_SO_VCN,KL,1000.0,0.470000,485918.279561,...,NaN,NaN,NaN,NaN,1LTR,NaN,NaN,NaN,NaN,NaN
84447,Amazon ARIPL,B0FJRZP6JW,6157.0,731273,SF OATS 1KG JAR,SAFF OATS,TO,1000.0,6.157000,126480.737807,...,NaN,NaN,NaN,NaN,1KG,NaN,NaN,NaN,NaN,NaN
84448,Amazon ARIPL,B0FS6GCRDG,34.0,721916,SAFF GOLD RC 4L JAR,SAFF GOLD,KL,4000.0,0.136000,137662.938527,...,NaN,NaN,NaN,NaN,4 LTR,NaN,NaN,NaN,NaN,NaN
84449,Amazon ARIPL,B0G1SKX4CM,6229.0,811179,SAFF CDPRS CNO 1L,SAF_CDPRS,KL,1000.0,6.229000,330000.000000,...,NaN,NaN,NaN,NaN,1 LTR,NaN,NaN,NaN,NaN,NaN


In [10]:
soh_base.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1'],
      dtype='object')

In [11]:
df['file'] = 'SOH - 01 Jun.xlsx'

In [12]:
final_df = pd.concat([soh_base, df], ignore_index=True)
final_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,vol_in_lit,vol in rum,indexrate,indexbpm,club sku,roum,roum divide,month,day,month.1
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87246,Amazon ARIPL,B0FS6GCRDG,2.0,721916,SAFF GOLD RC 4L JAR,SAFF GOLD,KL,4000.0,0.008000,137662.938527,...,NaN,NaN,NaN,NaN,4 LTR,NaN,NaN,NaN,NaN,NaN
87247,Amazon ARIPL,B0FS6P6VTW,4.0,721914,SAF GOLD 4L JAR,SAFF GOLD,KL,4000.0,0.016000,137662.938527,...,NaN,NaN,NaN,NaN,4 LTR,NaN,NaN,NaN,NaN,NaN
87248,Amazon ARIPL,B0G1SKX4CM,5408.0,811179,SAFF CDPRS CNO 1L,SAF_CDPRS,KL,1000.0,5.408000,330000.000000,...,NaN,NaN,NaN,NaN,1 LTR,NaN,NaN,NaN,NaN,NaN
87249,Amazon ARIPL,B0GGHLBJL3,854.0,732780,SF HIGH PROTEIN OAT 1KG PC,SFOATS-PR,TO,1000.0,0.854000,430000.000000,...,NaN,NaN,NaN,NaN,1KG,NaN,NaN,NaN,NaN,NaN


In [13]:
final_df['date'] = pd.to_datetime(final_df['date'])
final_df['date'].unique()

<DatetimeArray>
['2025-04-01 00:00:00', '2025-03-28 00:00:00', '2025-03-11 00:00:00',
 '2025-03-29 00:00:00', '2025-03-23 00:00:00', '2025-03-22 00:00:00',
 '2025-03-31 00:00:00', '2025-09-05 00:00:00', '2025-09-01 00:00:00',
 '2025-08-26 00:00:00', '2025-09-02 00:00:00', '2025-07-28 00:00:00',
 '2025-08-02 00:00:00', '2025-10-03 00:00:00', '2025-10-01 00:00:00',
 '2025-10-06 00:00:00', '2025-10-05 00:00:00', '2025-09-28 00:00:00',
 '2025-10-07 00:00:00', '2025-09-14 00:00:00', '2025-09-12 00:00:00',
 '2025-09-15 00:00:00', '2025-09-10 00:00:00', '2025-09-16 00:00:00',
 '2025-08-17 00:00:00', '2025-08-18 00:00:00', '2025-08-12 00:00:00',
 '2025-08-08 00:00:00', '2025-02-24 00:00:00', '2025-02-20 00:00:00',
 '2025-02-23 00:00:00', '2025-02-22 00:00:00', '2025-02-04 00:00:00',
 '2025-02-17 00:00:00', '2025-06-19 00:00:00', '2025-06-27 00:00:00',
 '2025-06-23 00:00:00', '2025-06-24 00:00:00', '2025-06-02 00:00:00',
 '2025-01-19 00:00:00', '2025-01-09 00:00:00', '2025-01-25 00:00:00',
 '20

In [14]:
final_df.to_csv('/data/aman_singh/acuuracy_check/soh_base_jun_run.csv', index=False)

In [17]:
final_df['month_end'] = final_df['date'] + pd.offsets.MonthEnd(0)

In [18]:
final_df.groupby(['month_end'])['soh'].sum()

month_end
2024-12-31    2.695180e+06
2025-01-31    3.063929e+06
2025-02-28    2.716948e+06
2025-03-31    1.670496e+06
2025-04-30    4.745680e+06
2025-05-31    2.740214e+06
2025-06-30    4.214220e+06
2025-07-31    1.971143e+06
2025-08-31    7.356897e+06
2025-09-30    9.830427e+06
2025-10-31    7.956328e+06
2025-11-30    1.127015e+06
2025-12-31    7.708016e+06
2026-01-31    4.790708e+06
2026-02-28    2.795760e+06
2026-03-31    3.647053e+06
2026-04-30    4.082873e+06
2026-05-31    4.148106e+06
2026-06-30    4.404793e+06
Name: soh, dtype: float64

In [14]:
# def clean_date(x):
#     if isinstance(x, str):
#         x = x.replace('st', '').replace('nd', '').replace('rd', '').replace('th', '').strip()
#         # if 'dec' in x.lower():
#         #     x += ' 2024'
#         # else:
#         #     x += ' 2025'
#         x += ' 2025'
#         return pd.to_datetime(x, format='%d %b %Y')
#     else:
#         return pd.to_datetime(x, unit='D', origin='1899-12-30')

In [15]:
# df['date_clean'] = df['date'].map(clean_date)

In [16]:
# df[['date', 'date_clean']].drop_duplicates().to_clipboard(index=False)

In [11]:
# del df['date']

# df.rename(columns={'date_clean': 'date'}, inplace=True)
df['date'] = pd.to_datetime(df['date'])

In [12]:
df.dtypes

date             datetime64[ns]
chain                    object
fsn                      object
soh                       int64
material_code             int64
desc                     object
brand                    object
uom                      object
vol per unit            float64
vol                     float64
fy index                float64
vol in kl               float64
bpm in lacs             float64
category                 object
ecom brands              object
psku                    float64
pdes                     object
club sku                 object
dtype: object

In [13]:
df[['platform_name', 'date']].drop_duplicates().sort_values(by=['platform_name'])

KeyError: "['platform_name'] not in index"

In [20]:
# ###### !!!!!!!!!!!!!!! TEMP #################!!!!!!!!!!!!!!!!!
# print(df[df['date'] > '2025-12-31']['date'].unique())
# df['date'] = pd.to_datetime(df['date'])

# cap_date = pd.to_datetime('2025-12-31')
# df['date'] = df['date'].clip(upper=cap_date)

# print(df[df['date'] > '2025-12-31']['date'].unique())

In [21]:
df['date'].max()

Timestamp('2026-02-02 00:00:00')

In [22]:
df[['platform_name', 'date']].drop_duplicates()

,platform_name,date
0,Flipkart Grocery,2026-01-22
209,Nykaa,2026-01-21
362,Blinkit,2026-01-31
557,Zepto,2026-01-31
761,Meesho,2026-01-31
1134,Big Basket,2026-01-31
1306,Swiggy,2026-01-31
1586,Flipkart National,2026-01-30
2042,Flipkart Minutes,2026-01-30
2287,Myntra,2026-01-28


In [23]:
df.rename(columns={'platform_name': 'chain', 'shipped_unit': 'soh'}, inplace=True)

In [24]:
df.to_csv('Clean Master/SOH - 31-Jan26 1 - processed.csv', index=False)

In [11]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [12]:
df.columns = df.columns.str.upper()
df['DATE'] = df['DATE'].astype(str)

In [13]:
df.dtypes

CHAIN             object
FSN               object
SOH              float64
MATERIAL_CODE     object
DESC              object
BRAND             object
UOM               object
VOL PER UNIT     float64
VOL              float64
FY INDEX         float64
BPM IN LACS      float64
CATEGORY          object
ECOM BRANDS       object
FILE              object
VOL IN KL        float64
PSKU              object
PDES              object
DATE              object
dtype: object

In [14]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, df, 
            table_name = "TRN_ECOM_SOH_FROM_BASE_FILES",
            auto_create_table=True,
            overwrite = False,
)

ArrowInvalid: ("Could not convert '00128d5d-d85d-40ed-b70b-c5fa9f10cac0' with type str: tried to convert to int64", 'Conversion failed for column FSN with type object')